# Week 3 — JOINs: Connecting Tables
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Until now every query you wrote read from a single table. But the answer to a real business question usually lives in two places at once: `orders` knows the status of an order, `customers` knows which state that customer is in, and neither table knows what the other one knows. `JOIN` stitches them together on the column they share.

Each question below comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the cell then displays the table your query returned.

**Do not edit the check cells.** Run the setup cell first, then work top to bottom.

Reminders for this week:
- The join key goes in the `ON` clause: `FROM orders o JOIN customers c ON o.customer_id = c.customer_id`.
- Always give each table a short **alias** (`o`, `c`, `oi`, `s`) and prefix every column with it. Two tables can hold columns with the same name, and the alias is what tells SQLite which one you mean.
- `JOIN` and `INNER JOIN` mean exactly the same thing in SQLite — a row survives only if the key matches on **both** sides.
- The clause order never changes: `SELECT … FROM … JOIN … ON … WHERE … GROUP BY … ORDER BY … LIMIT`.
- Alias your aggregates exactly as each question asks (e.g. `AS order_count`) — the check cells look up those column names.


In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

## Question 1 — Your first join: orders and their customers

The `orders` table stores a `customer_id`, but it has no idea *where* that customer lives. The `customers` table holds `customer_state` and `customer_city`, keyed by the same `customer_id`. Join the two tables on that shared column and return the first **10** rows with these four columns, in this order:

- `order_id` and `order_status` (from `orders`)
- `customer_state` and `customer_city` (from `customers`)

Give `orders` the alias `o` and `customers` the alias `c`, and use `LIMIT` to keep the output to 10 rows.

**Expected:** 10 rows with the columns `order_id`, `order_status`, `customer_state`, `customer_city` — each order now carries its customer's location alongside it.


In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
assert q1.shape[0] == 10, f"Q1: expected 10 rows (use LIMIT 10), got {q1.shape[0]}"
for col in ['order_id', 'order_status', 'customer_state', 'customer_city']:
    assert col in q1.columns, f"Q1: missing the '{col}' column — check your SELECT list"
print("✅ Q1 correct")
q1  # show the result of your query

## Question 2 — Where do delivered orders come from?

Now put the join to work. Join `orders` to `customers`, keep only the rows where `order_status` is `'delivered'`, then group by `c.customer_state` and count the orders in each state. Alias the count as `order_count`, sort from the busiest state down, and return only the **top 5**.

Notice how the pieces stack: the `JOIN` brings the state into reach, `WHERE` filters the rows, `GROUP BY` collapses them per state, and `ORDER BY … DESC` + `LIMIT 5` picks the winners.

**Expected:** 5 rows — SP 40,501 | RJ 12,350 | MG 11,354 | RS 5,345 | PR 4,923


In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
assert q2.shape[0] == 5, f"Q2: expected 5 rows (use LIMIT 5), got {q2.shape[0]}"
assert list(q2['customer_state']) == ['SP', 'RJ', 'MG', 'RS', 'PR'], \
    "Q2: expected the states SP, RJ, MG, RS, PR in that order — sort by the count descending"
assert int(q2.iloc[0]['order_count']) == 40501, "Q2: expected 40,501 delivered orders in SP"
assert int(q2.iloc[2]['order_count']) == 11354, "Q2: expected 11,354 delivered orders in MG"
assert int(q2.iloc[4]['order_count']) == 4923, "Q2: expected 4,923 delivered orders in PR"
print("✅ Q2 correct")
q2  # show the result of your query

## Question 3 — One state, one number

Question 2 gave you a ranking. This time the business wants a single figure: **how many delivered orders came from customers in `MG`?**

Join `orders` to `customers` again, but instead of grouping, filter on *two* conditions at once — the order must be `'delivered'` **and** the customer's state must be `'MG'` — and return the count as a single row aliased `mg_delivered`. No `GROUP BY` is needed when you want one number for the whole filtered set.

**Expected:** 11,354 delivered orders from MG


In [ ]:
%%sql q3 <<
-- Your query here

In [ ]:
# --- CHECK Q3 — do not edit ---
assert q3.shape[0] == 1, f"Q3: expected a single row, got {q3.shape[0]} — you don't need GROUP BY here"
assert int(q3.iloc[0]['mg_delivered']) == 11354, "Q3: expected 11,354 delivered orders from MG"
print("✅ Q3 correct")
q3  # show the result of your query

## Question 4 — Revenue by seller state

Switch to the sell side. `order_items` has one row per item sold, carrying a `seller_id` and the item `price`; `sellers` maps each `seller_id` to a `seller_state`. Join them (alias `order_items` as `oi` and `sellers` as `s`), group by `s.seller_state`, and return for each state:

- the number of **distinct** sellers, aliased as `seller_count` — use `COUNT(DISTINCT oi.seller_id)`, because one seller appears on many item rows and a plain `COUNT(*)` would count them over and over
- the total item revenue rounded to 2 decimal places, aliased as `total_revenue`

Sort by `total_revenue` descending and return the **top 5** states.

**Expected:** 5 rows — SP 1,849 sellers / 8,753,396.21 | PR 349 / 1,261,887.21 | MG 244 / 1,011,564.74 | RJ 171 / 843,984.22 | SC 190 / 632,426.07


In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
assert q4.shape[0] == 5, f"Q4: expected 5 rows (use LIMIT 5), got {q4.shape[0]}"
assert list(q4['seller_state']) == ['SP', 'PR', 'MG', 'RJ', 'SC'], \
    "Q4: expected the states SP, PR, MG, RJ, SC in that order — sort by total_revenue descending"
assert int(q4.iloc[0]['seller_count']) == 1849, \
    "Q4: expected 1,849 distinct sellers in SP — did you use COUNT(DISTINCT oi.seller_id)?"
assert round(float(q4.iloc[0]['total_revenue']), 2) == 8753396.21, "Q4: expected SP revenue of 8,753,396.21"
assert round(float(q4.iloc[1]['total_revenue']), 2) == 1261887.21, "Q4: expected PR revenue of 1,261,887.21"
assert int(q4.iloc[4]['seller_count']) == 190, "Q4: expected 190 distinct sellers in SC"
print("✅ Q4 correct")
q4  # show the result of your query

## Question 5 — The ten biggest sellers

Same two tables as Question 4, but zoom in one level: instead of grouping by state, group by the individual seller. Join `order_items` (`oi`) to `sellers` (`s`) and return, for the **10** highest-earning sellers:

- `seller_id` and `seller_state`
- `items_sold` — the number of item rows for that seller (a plain `COUNT(*)` is right here, because each row *is* one item sold)
- `total_revenue` — the sum of `oi.price`, rounded to 2 decimal places

Group by both `oi.seller_id` and `s.seller_state` (every non-aggregated column in the `SELECT` list must appear in `GROUP BY`), sort by `total_revenue` descending, and keep the top 10.

**Expected:** 10 rows, ordered from the highest total revenue down, with the columns `seller_id`, `seller_state`, `items_sold`, `total_revenue`. The check cell verifies the shape and the ordering rather than a fixed figure — read the table and see which states the top sellers sit in.


In [ ]:
%%sql q5 <<
-- Your query here

In [ ]:
# --- CHECK Q5 — do not edit ---
assert q5.shape[0] == 10, f"Q5: expected 10 rows (use LIMIT 10), got {q5.shape[0]}"
for col in ['seller_id', 'seller_state', 'items_sold', 'total_revenue']:
    assert col in q5.columns, f"Q5: missing the '{col}' column — check your aliases"
revenues = [float(v) for v in q5['total_revenue']]
assert revenues == sorted(revenues, reverse=True), \
    "Q5: rows are not sorted from the highest total_revenue down — add ORDER BY total_revenue DESC"
assert q5['seller_id'].nunique() == 10, "Q5: expected 10 different sellers — group by oi.seller_id"
assert int(q5['items_sold'].min()) >= 1, "Q5: every seller should have at least one item sold"
print("✅ Q5 correct")
q5  # show the result of your query